# 1. Problem Statement & Goals 🎯
___
## Problem Statement
Ebuss, a growing e-commerce company with a significant market share in categories like household essentials, personal care, and electronics, aims to scale rapidly and compete with market leaders like Amazon and Flipkart.

To achieve this, Ebuss needs to leverage its vast data on user reviews and ratings. As a Senior Machine Learning Engineer, the core challenge is to build a sentiment-based product recommendation system. This system must not only recommend products based on user behaviors (ratings) but also refine those recommendations by analyzing the sentiment of the textual reviews associated with those products. The ultimate objective is to enhance the user experience by suggesting products that users are most likely to purchase and feel positive about.

## Goals
The project is divided into four main objectives to achieve the problem statement:

### 1. Data Sourcing and Sentiment Analysis

#### Objective: 
* Build a Machine Learning model to classify user reviews as Positive or Negative.

#### Key Tasks:

* Perform Exploratory Data Analysis (EDA), data cleaning, and text preprocessing.

* Extract features using techniques like Bag-of-Words, TF-IDF, or Word Embeddings.

* Train and evaluate at least three of the following classification models: Logistic Regression, Random Forest, XGBoost, or Naive Bayes.

* Select the best-performing model to predict user sentiment.

### 2. Building a Recommendation System

#### Objective: 
* Identify the most effective recommendation technique for the dataset.

#### Key Tasks:

* Develop both User-based and Item-based collaborative filtering recommendation systems.

* Analyze and compare their performance to select the best-suited system.

* Generate an initial list of 20 recommended products for a specific user based on their historical ratings.

### 3. Improving Recommendations using Sentiment Analysis

#### Objective: 
* Create a hybrid "Sentiment-Based Recommendation System."

#### Key Tasks:

* Integrate the chosen Sentiment Analysis model with the Recommendation System.

* Take the top 20 products recommended by the collaborative filtering system.

* Filter and rank these products based on their predicted sentiment scores.

* Output the final top 5 products that have the highest positive sentiment.

### 4. Deployment

#### Objective: 
* Make the solution accessible via a web interface.

#### Key Tasks:

* Build a web application using the Flask framework.

* Create a User Interface (UI) that accepts a username and displays the top 5 recommended products.

* Deploy the end-to-end application (Model + API + UI) on a cloud platform like Heroku.m

In [36]:
# Importing Required Libraries
import random
from pathlib import Path
import os

# Data Science Libraries
import pandas as pd
import numpy as np

# Notebook Setup
from notebook_setup import NotebookInitializer
# Pass the path of the current file to the initializer
initializer = NotebookInitializer(Path(os.getcwd()).resolve())
initializer.setup_environment()

ROOT_DIR already set to: D:\Projects\PRS
Original working directory: D:\Projects\PRS
Current working directory changed to the project root: D:\Projects\PRS

--- Directory Structure Setup ---
📂 Root Directory: D:\Projects\PRS
📁 Data Directory: D:\Projects\PRS\data
📥 Raw Data Directory: D:\Projects\PRS\data\raw
📤 Processed Data Directory: D:\Projects\PRS\data\processed
⚙️ Config Manager: ConfigManager(config_path=config.json, model_path=models/)


In [2]:
# Local Utils
from utils import DataFileManager

In [3]:
df = DataFileManager.load_csv_data(initializer.processed_data_dir / "df_final.csv")

Loading data from D:\Projects\PRS\data\processed\df_final.csv
Data successfully loaded. Shape: (29877, 11)


In [5]:
df.columns

Index(['brand', 'categories', 'manufacturer', 'product_name', 'reviews_date',
       'reviews_doRecommend', 'reviews_rating', 'reviews_text',
       'reviews_title', 'reviews_username', 'user_sentiment'],
      dtype='object')

In [23]:
ratings = df[['reviews_username', 'product_name', 'reviews_rating']]
ratings

,reviews_username,product_name,reviews_rating
0,joshua,pink friday roman reloaded w dvd,5
1,dorothy w,lundberg organic cinnamon toast rice cakes,5
2,dorothy w,lundberg organic cinnamon toast rice cakes,5
3,rebecca,k y love sensuality pleasure gel,1
4,walker557,k y love sensuality pleasure gel,1
...,...,...,...
29872,laurasnchz,l'or233al paris elvive extraordinary clay reba...,5
29873,scarlepadilla,l'or233al paris elvive extraordinary clay reba...,5
29874,liviasuexo,l'or233al paris elvive extraordinary clay reba...,5
29875,ktreed95,l'or233al paris elvive extraordinary clay reba...,5


In [29]:
ratings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29877 entries, 0 to 29876
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   reviews_username  29877 non-null  object
 1   product_name      29877 non-null  object
 2   reviews_rating    29877 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 700.4+ KB


In [24]:
len(ratings['reviews_username'].unique())

24859

In [25]:
len(ratings['product_name'].unique())

268

# 2. Dividing the dataset into train and test

In [100]:
# Test and Train split of the dataset.
from sklearn.model_selection import train_test_split
train, test = train_test_split(ratings, test_size=0.30, random_state=31)

In [27]:
print(train.shape)
print(test.shape)

(20913, 3)
(8964, 3)


In [28]:
train.head()

,reviews_username,product_name,reviews_rating
12684,kris101,clorox disinfecting wipes value pack scented 1...,5
21969,activegal,yes grapefruit rejuvenating body wash,5
24634,philippe,godzilla 3d includes digital copy ultraviolet ...,5
26224,bbshopper,stargate ws ultimate edition director cut dvdv...,5
26853,georgew,jason aldean know,5


## Using adjusted Cosine 
### Here, we are not removing the NaN values and calculating the mean only for the movies rated by the user

In [87]:
# Pivot the train ratings' dataset into matrix format in which columns are movies and the rows are user IDs.
df_pivot = train.pivot_table(
    index='reviews_username',
    columns='product_name',
    values='reviews_rating',
    aggfunc='mean'
).T

df_pivot.head(3)

reviews_username,00dog3,00sab00,01impala,02dakota,02deuce,0325home,06stidriver,09mommy11,1085,10ten,...,zsarah,zsazsa,zt313,zubb,zuttle,zwithanx,zxcsdfd,zyiah4,zzdiane,zzz1127
product_name,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100 complete season blu ray,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017 2018 brownline174 duraflex 14 month planner 8 1/2 x 11 black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Normalising the rating of the movie for each user around 0 mean

In [88]:
mean = np.nanmean(df_pivot, axis=1)
df_subtracted = (df_pivot.T-mean).T

In [89]:
df_subtracted.head()

reviews_username,00dog3,00sab00,01impala,02dakota,02deuce,0325home,06stidriver,09mommy11,1085,10ten,...,zsarah,zsazsa,zt313,zubb,zuttle,zwithanx,zxcsdfd,zyiah4,zzdiane,zzz1127
product_name,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
100 complete season blu ray,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2017 2018 brownline174 duraflex 14 month planner 8 1/2 x 11 black,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2x ultra era oxi booster 50fl oz,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4c grated parmesan cheese 100 natural 8 oz shaker,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [90]:
from sklearn.metrics.pairwise import pairwise_distances

In [91]:
# Creating the User Similarity Matrix using pairwise_distance function.
user_correlation = 1 - pairwise_distances(df_subtracted.fillna(0), metric='cosine')
user_correlation[np.isnan(user_correlation)] = 0
print(user_correlation)

[[ 1.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          1.          0.         ... -0.00839544  0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 ...
 [ 0.         -0.00839544  0.         ...  1.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          1.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   1.        ]]


## Prediction - User User

Doing the prediction for the users which are positively related with other users, and not the users which are negatively related as we are interested in the users which are more similar to the current users. So, ignoring the correlation for values less than 0. 

In [92]:
user_correlation[user_correlation<0]

array([-1.58323786e-01, -9.32592652e-03, -1.70197733e-02, -5.34666202e-03,
       -7.61932482e-04, -4.41482436e-03, -2.83101603e-03, -8.39543834e-03,
       -3.68371093e-03, -7.39146151e-03, -1.16042437e-04, -5.23312680e-03,
       -3.11274704e-02, -1.20777160e-03, -1.19301328e-03, -1.26900956e-02,
       -5.90965362e-03, -4.88096517e-04, -1.40917739e-03, -2.56972763e-02,
       -2.29494366e-02, -2.76078815e-01, -8.80418106e-04, -3.93565574e-03,
       -3.33357264e-02, -7.67898832e-02, -9.89689068e-03, -8.83860462e-03,
       -1.09206932e-04, -1.06265554e-03, -1.06327294e-01, -7.20127180e-03,
       -9.89041063e-04, -6.02894101e-04, -1.87329777e-03, -8.90547812e-06,
       -1.64879954e-03, -1.02497085e-03, -2.08028846e-03, -8.07463199e-04,
       -1.59086460e-02, -1.03188468e-03, -1.04709838e-03, -4.96668463e-03,
       -1.42496254e-03, -8.23225372e-04, -2.63523435e-03, -1.60637137e-03,
       -3.55358111e-02, -1.16029883e-03, -2.08480816e-03, -1.00963836e-03,
       -1.20916097e-03, -

In [93]:
user_correlation[user_correlation<0]=0
user_correlation

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]], shape=(251, 251))

Rating predicted by the user (for movies rated as well as not rated) is the weighted sum of correlation with the movie rating (as present in the rating dataset). 

In [94]:
user_predicted_ratings = np.dot(user_correlation, df_pivot.fillna(0))
user_predicted_ratings

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.88363165e-02, ...,
        1.65268791e-03, 0.00000000e+00, 1.32215032e-03],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [2.71935399e-02, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.26988128e-05, 2.25384438e-03, ...,
        0.00000000e+00, 5.44980214e-05, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 1.23029105e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],
      shape=(251, 18231))

In [95]:
user_predicted_ratings.shape

(251, 18231)

In [96]:
user_predicted_ratings[112]

array([0., 0., 0., ..., 0., 0., 0.], shape=(18231,))

# Evaluation - User User 
Evaluation will we same as you have seen above for the prediction. The only difference being, you will evaluate for the movie already rated by the user insead of predicting it for the movie not rated by the user. 

In [101]:
test

,reviews_username,product_name,reviews_rating
18433,chuck7,delta single handle shower faucet,5
29411,vyph,l'or233al paris elvive extraordinary clay reba...,4
10653,lia02,clorox disinfecting wipes value pack scented 1...,5
16497,rubyred,burt bees lip shimmer raisin,5
283,kittyc,olay regenerist deep hydration regenerating cream,5
...,...,...,...
20643,devin holland,head shoulders classic clean conditioner,5
29549,heather f,l'or233al paris elvive extraordinary clay reba...,5
26579,bysgedgaudas,aveeno baby continuous protection lotion sunsc...,5
19925,justamom,clorox disinfecting bathroom cleaner,5


In [102]:
# # Find out the common users of test and train dataset.
# common = test[test.reviews_username.isin(train.reviews_username)]
# common.shape

# Find out the common users of test and train dataset.
common = test[test.product_name.isin(train.product_name)]
common.shape

(8943, 3)

In [103]:
common.head()

,reviews_username,product_name,reviews_rating
18433,chuck7,delta single handle shower faucet,5
29411,vyph,l'or233al paris elvive extraordinary clay reba...,4
10653,lia02,clorox disinfecting wipes value pack scented 1...,5
16497,rubyred,burt bees lip shimmer raisin,5
283,kittyc,olay regenerist deep hydration regenerating cream,5


In [104]:
# convert into the user-movie matrix.
common_user_based_matrix = common.pivot_table(index='reviews_username',
    columns='product_name',
    values='reviews_rating'
).T

In [105]:
# Convert the user_correlation matrix into dataframe.
user_correlation_df = pd.DataFrame(user_correlation)

In [106]:
df_subtracted.head(1)

reviews_username,00dog3,00sab00,01impala,02dakota,02deuce,0325home,06stidriver,09mommy11,1085,10ten,...,zsarah,zsazsa,zt313,zubb,zuttle,zwithanx,zxcsdfd,zyiah4,zzdiane,zzz1127
product_name,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
user_correlation_df['reviews_username'] = df_subtracted.index
user_correlation_df.set_index('reviews_username',inplace=True)
user_correlation_df.head()

,0,1,2,3,4,5,6,7,8,9,...,241,242,243,244,245,246,247,248,249,250
reviews_username,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100 complete season blu ray,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017 2018 brownline174 duraflex 14 month planner 8 1/2 x 11 black,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2x ultra era oxi booster 50fl oz,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4c grated parmesan cheese 100 natural 8 oz shaker,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [108]:
common.head(1)

,reviews_username,product_name,reviews_rating
18433,chuck7,delta single handle shower faucet,5


In [114]:
# list_name = common.reviews_username.tolist()

# user_correlation_df.columns = df_subtracted.index.tolist()


# user_correlation_df_1 =  user_correlation_df[user_correlation_df.index.isin(list_name)]

list_name = common.product_name.tolist()

user_correlation_df.columns = df_subtracted.index.tolist()


user_correlation_df_1 =  user_correlation_df[user_correlation_df.index.isin(list_name)]

In [115]:
user_correlation_df_1.shape

(194, 251)

In [116]:
user_correlation_df_2 = user_correlation_df_1.T[user_correlation_df_1.T.index.isin(list_name)]

In [117]:
user_correlation_df_3 = user_correlation_df_2.T

In [118]:
user_correlation_df_3.head()

,0.6 cu ft letter a4 size waterproof 30 min fire file chest,100 complete season blu ray,2017 2018 brownline174 duraflex 14 month planner 8 1/2 x 11 black,2x ultra era oxi booster 50fl oz,4c grated parmesan cheese 100 natural 8 oz shaker,africa best lye dual conditioning relaxer system super,alberto vo5 salon series smooth plus sleek shampoo,alex cross dvdvideo,annie homegrown gluten free double chocolate chip granola bars,arrid extra dry anti perspirant deodorant spray regular,...,vaseline intensive care healthy hands stronger nails,vaseline intensive care lip therapy cocoa butter,vicks vaporub regular 3.53 oz,wagan smartac 80watt inverter usb,way basics 3 shelf eco narrow bookcase storage shelf espresso formaldehyde free lifetime guarantee,weathertech 40647 14 15 outlander cargo liners 2nd row black,weleda everon lip balm,windex original glass cleaner refill 67.6 oz 2 liter,yes carrots nourishing body wash,yes grapefruit rejuvenating body wash
reviews_username,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100 complete season blu ray,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2017 2018 brownline174 duraflex 14 month planner 8 1/2 x 11 black,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2x ultra era oxi booster 50fl oz,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4c grated parmesan cheese 100 natural 8 oz shaker,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [119]:
user_correlation_df_3.shape

(194, 194)

In [120]:
user_correlation_df_3[user_correlation_df_3<0]=0

common_user_predicted_ratings = np.dot(user_correlation_df_3, common_user_based_matrix.fillna(0))
common_user_predicted_ratings

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [5.00000000e+00, 1.65268791e-03, 1.32215032e-03, ...,
        0.00000000e+00, 1.77253489e-02, 6.61075162e-04],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        5.44980214e-05, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00]],
      shape=(194, 8341))

In [121]:
dummy_test = common.copy()

dummy_test['reviews_rating'] = dummy_test['reviews_rating'].apply(lambda x: 1 if x>=1 else 0)

dummy_test = dummy_test.pivot_table(index='reviews_username', columns='product_name', values='reviews_rating').T.fillna(0)

In [122]:
dummy_test.shape

(194, 8341)

In [123]:
common_user_predicted_ratings = np.multiply(common_user_predicted_ratings,dummy_test)

In [124]:
common_user_predicted_ratings.head(2)

reviews_username,08dallas,1.11E+24,1234,1234asdf,123numbers,127726,12gage,13dani,13ld,13ram,...,zkondrk,zmom,zoeellasca,zoey,zoeyny,zokhid,zombiejess,zombiekiller,zulaa118,zxjki
product_name,,,,,,,,,,,,,,,,,,,,,
0.6 cu ft letter a4 size waterproof 30 min fire file chest,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
100 complete season blu ray,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [125]:
from sklearn.preprocessing import MinMaxScaler
from numpy import *

X  = common_user_predicted_ratings.copy() 
X = X[X>0]

scaler = MinMaxScaler(feature_range=(1, 5))
print(scaler.fit(X))
y = (scaler.transform(X))

print(y)

MinMaxScaler(feature_range=(1, 5))
[[nan nan nan ... nan nan nan]
 [ 1. nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 ...
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]
 [nan nan nan ... nan nan nan]]


d:\Projects\PRS\.venv-notebook\Lib\site-packages\sklearn\utils\_array_api.py:686: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
d:\Projects\PRS\.venv-notebook\Lib\site-packages\sklearn\utils\_array_api.py:706: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))


In [126]:
# index='reviews_username',
#     columns='product_name',
#     values='reviews_rating',
#     aggfunc='mean'
common_ = common.pivot_table(index='reviews_username', columns='product_name', values='reviews_rating', aggfunc='mean').T

In [127]:
# Finding total non-NaN value
total_non_nan = np.count_nonzero(~np.isnan(y))

In [128]:
rmse = (sum(sum((common_ - y )**2))/total_non_nan)**0.5
print(rmse)

3.564767241087814


d:\Projects\PRS\.venv-notebook\Lib\site-packages\numpy\_core\fromnumeric.py:84: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
